In [1]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      6
    Socket(s):               1
    Stepping:                7
    BogoMIPS:                4400.45
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                 

In [2]:
!nvidia-smi

Sat Apr 25 11:21:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

PyTorch: 2.10.0+cu128
CUDA available: True
cuDNN version: 91002


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


# ReBRAC Stage D Phase 1 Step 2 — `worldcomp-1000` deployable formal

**目的**：在 Stage D Phase 1 Step 1 epoch-probe 已经把 `TRAIN_EPOCHS=64` 钉死、并在 2 seed × test=40 上观察到 `success_rate = 1.0 ± 0.0` 之后，按 5-seed × test=100 的正式协议重新复核——既扩 seed 数到 5（捕捉新 seed 的方差），又把 test manifest 升到 100 episodes（与 Stage C 和 TD3BC worldcomp formal 同口径）。

**核心问题**：ReBRAC 在 `worldcomp-1000` deployable 下是否打破了 TD3BC 退化为 BC 的现象（`success > 0.858`，并 `std ≤ 0.10`）？

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 / β2 | `4.0 / 2.0` | Stage C 锁定的 finalist，Stage D 不扫超参 |
| dataset | `worldcomp-1000` | Stage D 的核心 dataset |
| seeds | `42 43 44 45 46` | 与 Stage C 和 TD3BC worldcomp teacher-gap formal 对齐 |
| TRAIN_EPOCHS | `64` | 由 Stage D Phase 1 epoch-probe 决定（详见 [plan §6.5.2](../docs/rebrac_experiment_plan.md)） |
| 轨道 | deployable | actor + critic 都用 deployable obs；privileged-critic 是 Phase 2 |
| val manifest | 40 episodes | 与 Stage C 对齐 |
| test manifest | **100 episodes** | 与 Stage C / TD3BC worldcomp formal 同口径 |

**预算**：1 cell × 1 dataset × 5 seed = 5 个 train run。如果 worldcomp 数据已经被 epoch-probe collect 过，直接复用；否则 ensure 步骤会自动跳过。

**输出树**：
- `checkpoints/offline/rebrac/worldcomp_teacher_gap/deployable/`
- `results/offline/rebrac/worldcomp_teacher_gap/deployable/`

**Phase 1 通过判据 + Phase 2 触发情景**（详见 [plan §6.5.2](../docs/rebrac_experiment_plan.md)）：

| 情景 | Phase 1 deployable mean | Phase 2 形态 |
| --- | --- | --- |
| **A** 显著超 TD3BC | `> 0.90` | privileged-critic 3-seed 边际确认（或可省略） |
| **B** 与 TD3BC 持平 | `0.84 ~ 0.90` | privileged-critic 5-seed 完整诊断 |
| **C** 弱于 TD3BC | `< 0.84` | privileged-critic 5-seed + critic-penalty-off ablation |

probe 已经强烈暗示落在 **情景 A**（probe 2 seeds × test=40 拿到 `1.0 ± 0.0`），但需要 5 seeds × test=100 正式复核。


## 0. 环境配置


In [5]:
import os

# —— 通用（与其它 ReBRAC / TD3BC 实验一致）——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— Stage C 锁定的 finalist（Stage D 不扫超参）——
os.environ["ACTOR_PENALTY_COEF"]  = "4.0"
os.environ["CRITIC_PENALTY_COEF"] = "2.0"

# —— Phase 1 deployable formal 协议（覆盖 driver 默认）——
os.environ["DEPLOYABLE_FINAL_SEEDS"]                       = "42 43 44 45 46"
os.environ["DEPLOYABLE_FINAL_TRAIN_EPOCHS"]                = "64"   # 由 epoch-probe 决定
os.environ["DEPLOYABLE_FINAL_CHECKPOINT_EVERY_EPOCHS"]     = "8"
os.environ["FINAL_VAL_MANIFEST_EPISODES"]                  = "40"
os.environ["FINAL_TEST_MANIFEST_EPISODES"]                 = "100"  # apples-to-apples vs TD3BC worldcomp formal

# 其余（DATASET_POLICY=worldcomp, DATASET_EPISODES_VALUE=1000,
# BENCHMARK_KEY=single_u10_cross_tgt15, PROBE_LAYOUT=s0,
# HISTORY_LENGTH=4, TASK_GEOMETRY=cross_stream, TARGET_SPEED=1.5,
# OBJECTIVE=efficiency_v2, SAMPLING_MODE=shuffle_no_replacement,
# BATCH_SIZE=256）使用 driver 默认值。


## 1. 生成 val / test manifest（test=100）

driver 会用一个新的 manifest root（`benchmarks/offline_rebrac_worldcomp_final`），与 epoch-probe 隔离。已经存在的文件会被跳过。


In [6]:
os.environ["MODE"] = "deployable_manifests"
!bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh



[cmd] env MODE=manifests PYTHON_BIN=python3 DEVICE=cuda BENCHMARK_KEY=single_u10_cross_tgt15 FLOW_PATH=wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy TASK_GEOMETRY=cross_stream TARGET_SPEED=1.5 OBJECTIVE=efficiency_v2 PROBE_LAYOUT=s0 HISTORY_LENGTH=4 DATASET_POLICY=worldcomp DATASET_EPISODES=1000 DATASET_SEED=0 COLLECT_WORKERS=8 ACTOR_PENALTY_COEFS=4.0 CRITIC_PENALTY_COEFS=2.0 BATCH_SIZE=256 DROP_LAST_BATCH=0 SAMPLING_MODE=shuffle_no_replacement HIDDEN_DIM=256 NUM_HIDDEN_LAYERS=3 ACTOR_LR=3e-4 CRITIC_LR=3e-4 GAMMA=0.99 TAU=0.005 POLICY_NOISE=0.2 NOISE_CLIP=0.5 POLICY_FREQ=2 GRAD_CLIP_NORM=10.0 NORMALIZER_EPS=1e-3 LOG_EVERY=1000 TRAIN_METRICS_WINDOW_FRACTION=0.25 EVAL_WORKERS=6 EVAL_WORKER_DEVICE=cpu VALIDATION_SEED=123 TEST_SEED=456 FORCE_REEVAL=0 SEEDS=42 43 44 45 46 TRAIN_EPOCHS=64 CHECKPOINT_EVERY_EPOCHS=8 VAL_MANIFEST_EPISODES=40 TEST_MANIFEST_EPISODES=100 MANIFEST_ROOT=benchmarks/offline_rebrac_worldcomp_final CHECKPOINT_ROOT=checkpoints/offline/rebrac/worldcomp

## 2. 复用 worldcomp-1000 离线数据

`worldcomp-1000` 数据集如果在 epoch-probe 阶段已经 collect 过（位于 `offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/`），这里会直接复用。否则会自动触发一次 `MODE=deployable_manifests` 子命令引导的 collect。

为了显式确保复用而不重收数据，下面这步只是 sanity check——`screen.sh` 的 `ensure_dataset` 会打印 `[skip] dataset exists`。


In [7]:
import pathlib
dataset_dir = pathlib.Path("offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000")
if (dataset_dir / "transitions.npz").exists():
    print(f"[reuse] worldcomp-1000 数据已存在：{dataset_dir / 'transitions.npz'}")
else:
    print(f"[warn] {dataset_dir} 不存在；下一个 cell 的 train 步骤会自动调 collect")


[reuse] worldcomp-1000 数据已存在：offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000/transitions.npz


## 3. 全流程：train → validate（每个 ckpt）→ select → test → summarize

5 个 train run（5 seeds × 1 finalist × 64 epoch），每个产出 8 个 ckpt（每 8 epoch 一个），每个 ckpt 上跑 val=40。selection 按 `success_rate → return → -safety_cost → -time` 选最佳，最佳 ckpt 在 test=100 上重跑一次作为该 (seed) 的最终成绩。


In [ ]:
# 跑 Phase 1 deployable 全流程
for mode in ["deployable_train", "deployable_validate", "deployable_test", "deployable_summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh



[cmd] env MODE=train PYTHON_BIN=python3 DEVICE=cuda BENCHMARK_KEY=single_u10_cross_tgt15 FLOW_PATH=wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy TASK_GEOMETRY=cross_stream TARGET_SPEED=1.5 OBJECTIVE=efficiency_v2 PROBE_LAYOUT=s0 HISTORY_LENGTH=4 DATASET_POLICY=worldcomp DATASET_EPISODES=1000 DATASET_SEED=0 COLLECT_WORKERS=8 ACTOR_PENALTY_COEFS=4.0 CRITIC_PENALTY_COEFS=2.0 BATCH_SIZE=256 DROP_LAST_BATCH=0 SAMPLING_MODE=shuffle_no_replacement HIDDEN_DIM=256 NUM_HIDDEN_LAYERS=3 ACTOR_LR=3e-4 CRITIC_LR=3e-4 GAMMA=0.99 TAU=0.005 POLICY_NOISE=0.2 NOISE_CLIP=0.5 POLICY_FREQ=2 GRAD_CLIP_NORM=10.0 NORMALIZER_EPS=1e-3 LOG_EVERY=1000 TRAIN_METRICS_WINDOW_FRACTION=0.25 EVAL_WORKERS=6 EVAL_WORKER_DEVICE=cpu VALIDATION_SEED=123 TEST_SEED=456 FORCE_REEVAL=0 SEEDS=42 43 44 45 46 TRAIN_EPOCHS=64 CHECKPOINT_EVERY_EPOCHS=8 VAL_MANIFEST_EPISODES=40 TEST_MANIFEST_EPISODES=100 MANIFEST_ROOT=benchmarks/offline_rebrac_worldcomp_final CHECKPOINT_ROOT=checkpoints/offline/rebrac/worldcomp_tea

## 4. 主结果：5-seed test overview + per-seed 分布

读 `results/offline/rebrac/worldcomp_teacher_gap/deployable/.../summaries/overview.{csv,json}` 与 per-seed `test/seed_*.json` 拼成最终 5-seed 表。


In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path("results/offline/rebrac/worldcomp_teacher_gap/deployable")
DATASET_NAME = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
PAIR_TAG     = "actorb_4p0__criticb_2p0"
SEEDS        = os.environ["DEPLOYABLE_FINAL_SEEDS"].split()


def load_test_per_seed() -> pd.DataFrame:
    rows = []
    for seed in SEEDS:
        path = RESULTS_ROOT / DATASET_NAME / PAIR_TAG / "test" / f"seed_{seed}.json"
        if not path.exists():
            print(f"[warn] missing test json: {path}")
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "success_rate": payload["eval_success_rate"],
            "return": payload["eval_return"],
            "safety_cost": payload["eval_safety_cost"],
            "time_s": payload["eval_time_s"],
            "progress_ratio": payload.get("eval_progress_ratio"),
            "path_efficiency": payload.get("eval_path_efficiency"),
        })
    return pd.DataFrame(rows)


per_seed = load_test_per_seed()
print("[per-seed test results]")
print(per_seed.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

if not per_seed.empty:
    print()
    print("[summary]")
    print(f"  mean test success_rate = {per_seed['success_rate'].mean():.4f}")
    print(f"  std  test success_rate = {per_seed['success_rate'].std():.4f}")
    print(f"  mean test return       = {per_seed['return'].mean():.3f}")
    print(f"  std  test return       = {per_seed['return'].std():.3f}")
    print(f"  mean test safety_cost  = {per_seed['safety_cost'].mean():.3f}")


[per-seed test results]
seed  success_rate  return  safety_cost  time_s  progress_ratio  path_efficiency
  42        0.9900 26.8458       6.1532 54.3370          0.9015           0.7738
  43        0.9300 17.7562       6.8171 54.8420          0.8844           0.7666
  44        0.7800 -2.5275       8.1631 55.3240          0.8636           0.7253
  45        0.9800 28.9739       5.0199 52.7960          0.9007           0.7889
  46        0.9600 29.7970       3.6655 51.2100          0.8961           0.8023

[summary]
  mean test success_rate = 0.9280
  std  test success_rate = 0.0858
  mean test return       = 20.169
  std  test return       = 13.562
  mean test safety_cost  = 5.964


In [ ]:
# 读 overview summary，拿到 critic_penalty / target_q 诊断
import csv

overview_path = RESULTS_ROOT / "summaries" / "overview.csv"
if overview_path.exists():
    with overview_path.open(encoding="utf-8") as fp:
        reader = csv.DictReader(fp)
        for row in reader:
            print(f"[overview] dataset={row['dataset']} pair={row['pair']} num_seeds={row['num_seeds']}")
            print(f"  mean_test_success_rate = {float(row['mean_test_success_rate']):.4f}")
            print(f"  std_test_success_rate  = {float(row['std_test_success_rate']):.4f}")
            print(f"  mean_test_return       = {float(row['mean_test_return']):.3f}")
            print(f"  mean_critic_penalty    = {float(row['mean_critic_penalty']):.4f}")
            print(f"  mean_target_q          = {float(row['mean_target_q']):.3f}")
            penalty_ratio = float(row['mean_critic_penalty_ratio'])
            beta2 = float(os.environ['CRITIC_PENALTY_COEF'])
            print(f"  mean_critic_penalty_ratio = {penalty_ratio:.4f}  (β2 · ratio = {beta2 * penalty_ratio:.4f})")
else:
    print(f"[warn] overview.csv missing at {overview_path}")


[overview] dataset=worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000 pair=actorb_4p0__criticb_2p0 num_seeds=5
  mean_test_success_rate = 0.9280
  std_test_success_rate  = 0.0768
  mean_test_return       = 20.169
  mean_critic_penalty    = 0.0802
  mean_target_q          = 15.220
  mean_critic_penalty_ratio = 0.0055  (β2 · ratio = 0.0111)


## 5. 与 TD3BC worldcomp formal 三向对比

| 协议 | success (mean ± std) | return | safety_cost |
| --- | ---: | ---: | ---: |
| TD3BC `worldcomp` deployable formal (α=0.0, BC) | `0.858 ± 0.080` | `−14.29` | `7.91` |
| TD3BC `worldcomp` privileged-critic formal (α=0.1) | `0.922 ± 0.086` | `16.49` | `5.73` |
| teacher baseline (online) | `0.990` | `32.19` | — |
| **ReBRAC `worldcomp` deployable Phase 1**（本 notebook） | （上面 cell 计算）| | |

读完 cell 11 / 12 之后，把数字填回这个表（人工，或下个 cell 自动算）。


In [ ]:
if not per_seed.empty:
    rebrac_mean = per_seed['success_rate'].mean()
    rebrac_std  = per_seed['success_rate'].std()
    rebrac_ret  = per_seed['return'].mean()
    rebrac_safe = per_seed['safety_cost'].mean()

    td3bc_dep_mean = 0.858
    td3bc_priv_mean = 0.922
    teacher_mean = 0.990

    print("=" * 70)
    print(f"{'Protocol':<46}{'mean':>8}{'std':>8}")
    print("-" * 70)
    print(f"{'TD3BC worldcomp deployable (α=0.0, BC)':<46}{td3bc_dep_mean:>8.3f}{0.080:>8.3f}")
    print(f"{'TD3BC worldcomp privileged-critic (α=0.1)':<46}{td3bc_priv_mean:>8.3f}{0.086:>8.3f}")
    print(f"{'teacher baseline (online)':<46}{teacher_mean:>8.3f}{'-':>8}")
    print(f"{'ReBRAC worldcomp deployable (β1=4, β2=2)':<46}{rebrac_mean:>8.4f}{rebrac_std:>8.4f}")
    print("=" * 70)
    print()

    delta_dep = rebrac_mean - td3bc_dep_mean
    delta_priv = rebrac_mean - td3bc_priv_mean
    delta_teacher = rebrac_mean - teacher_mean

    print(f"Δ vs TD3BC deployable     = {delta_dep:+.4f} ({delta_dep*100:+.1f}pp)")
    print(f"Δ vs TD3BC privileged     = {delta_priv:+.4f} ({delta_priv*100:+.1f}pp)")
    print(f"Δ vs teacher baseline     = {delta_teacher:+.4f} ({delta_teacher*100:+.1f}pp)")

    # Gap closure relative to teacher baseline (treat teacher as ceiling at 0.990)
    deployable_gap = teacher_mean - td3bc_dep_mean
    rebrac_gap_closed = (rebrac_mean - td3bc_dep_mean) / deployable_gap
    print(f"\nGap closure (vs TD3BC deployable → teacher):")
    print(f"  TD3BC privileged-critic = {(td3bc_priv_mean - td3bc_dep_mean) / deployable_gap * 100:.1f}%")
    print(f"  ReBRAC  deployable      = {rebrac_gap_closed * 100:.1f}%")

    # Phase 2 scenario
    print()
    if rebrac_mean > 0.90:
        scenario = "A (显著超 TD3BC) → Phase 2 privileged-critic 3-seed 边际确认（或可省略）"
    elif rebrac_mean >= 0.84:
        scenario = "B (与 TD3BC 持平) → Phase 2 privileged-critic 5-seed 完整诊断"
    else:
        scenario = "C (弱于 TD3BC) → Phase 2 privileged-critic 5-seed + critic-penalty-off ablation"
    print(f"[Phase 2 触发情景] {scenario}")


Protocol                                          mean     std
----------------------------------------------------------------------
TD3BC worldcomp deployable (α=0.0, BC)           0.858   0.080
TD3BC worldcomp privileged-critic (α=0.1)        0.922   0.086
teacher baseline (online)                        0.990       -
ReBRAC worldcomp deployable (β1=4, β2=2)        0.9280  0.0858

Δ vs TD3BC deployable     = +0.0700 (+7.0pp)
Δ vs TD3BC privileged     = +0.0060 (+0.6pp)
Δ vs teacher baseline     = -0.0620 (-6.2pp)

Gap closure (vs TD3BC deployable → teacher):
  TD3BC privileged-critic = 48.5%
  ReBRAC  deployable      = 53.0%

[Phase 2 触发情景] A (显著超 TD3BC) → Phase 2 privileged-critic 3-seed 边际确认（或可省略）


## 6. Phase 1 通过判据核对

按 [plan §6.5.2 Phase 1 通过判据](../docs/rebrac_experiment_plan.md)：

| 条件 | 阈值 | Phase 1 实测 | 是否通过 |
| --- | --- | --- | --- |
| Phase 1 deployable mean test success | `> 0.858` (TD3BC deployable formal) | （上 cell 计算） | |
| Phase 1 deployable std test success | `≤ 0.10` | （上 cell 计算） | |

跑完上面 cell 后，把判定结果写入 `docs/rebrac_experiment_report.md` 的 Stage D Phase 1 deployable 小节，并据此决定 Phase 2 (`rebrac_worldcomp_phase2_privileged.ipynb`) 的 `PRIVILEGED_FINAL_SEEDS`：
- 情景 A：`PRIVILEGED_FINAL_SEEDS="42 43 44"`（3-seed 边际确认）
- 情景 B / C：`PRIVILEGED_FINAL_SEEDS="42 43 44 45 46"`（5-seed 完整）
- 情景 A 极端版：直接跳过 Phase 2，记录决策依据
